# Questão 1

In [ ]:
publicacoes_brutas = pd.read_csv('dados/publicacoes_brutas.csv')
print(f"Linhas: {publicacoes_brutas.shape[0]}, colunas: {publicacoes_brutas.shape[1]}")
print(publicacoes_brutas[["id_publicacao",	"data_publicacao",	"tema",	"formato",	"alcance",	"curtidas",	"comentarios",	"compartilhamentos",	"salvamentos",	"seguidores_autor",	"videos_autor",	"duracao_segundos"]].isna().sum()) 
publicacoes_brutas.head(5)

Os resultados tratados a partir desses dados dizem respeito somente a esta coleta em específico, com dados coletados especificamente dos posts da campanha da empresa, e não podem ser tratadas como um retrato de todas as redes, públicos ou bairros.

# Questão 2

In [ ]:
df = pd.read_csv('dados/publicacoes_brutas.csv')
df = df.drop_duplicates(subset='id_publicacao')
linhas_apos_dedup = len(df)
print(f"Linhas após remover duplicatas: {linhas_apos_dedup}")

df["tema"] = df["tema"].str.strip().str.title()

temas_equivalentes = {"Saude": "Saúde"}  # mapeamento manual: grafias diferentes, mesmo conceito
temas_unificadas = df["tema"].isin(temas_equivalentes.keys()).sum()  # quantas linhas usavam a grafia alternativa
df["tema"] = df["tema"].replace(temas_equivalentes)

df["data_publicacao"] = pd.to_datetime(
    df["data_publicacao"],
    format="mixed",  # aceita formatos diferentes linha a linha, em vez de exigir um único padrão fixo
    dayfirst=True,  # em caso de ambiguidade, o primeiro número do texto é o dia
    errors="coerce",  # data impossível de interpretar vira NaT, sem derrubar o processamento
)

datas_invalidas = df["data_publicacao"].isna().sum()

alcances_mantidos_ausentes = df["alcance"].isna().sum()  # preço ausente/não numérico, mantido como está, por decisão
salvamentos_mantidos_ausentes = df["salvamentos"].isna().sum()
comentarios_mantidos_ausentes = df["comentarios"].isna().sum()
print(f"Alcances, salvamentos e comentários mantidos como ausentes, pois assumir valor 0 atrapalharia a analise e remover a linha não é necessário (decisão documentada): {métricas_mantidos_ausentes, salvamentos_mantidos_ausentes, comentarios_mantidos_ausentes}")

df["taxa_utilidade_pct"] = (
    df["compartilhamentos"] + df["salvamentos"]
) / df["alcance"] * 100

mediana_tema = df.groupby("tema")["taxa_utilidade_pct"].agg(
    mediana_utilidade="median",
    n_postagens="count"
).reset_index()

mediana_tema = mediana_tema.sort_values("mediana_utilidade")

mediana_tema



In [ ]:
mediana_tema = df.groupby("tema")["taxa_utilidade_pct"].mean().reset_index()  # média da taxa de engajamento, por dia
mediana_tema = mediana_tema.sort_values("taxa_utilidade_pct")

mediana_tema

Primeiro removi as linhas duplicadas por id_publicação, padronizei os valores de tema com .str.title() e igualei Saude para Saúde, converti as datas para deixá-las padronizadas, com o primeiro número definido como dia, manti os valores ausentes de alcance, salvamentos e compartilhamentos, pois igualar eles a 0 indicaria uma métrica errônea e não removi a linha pois é possível analisar outras variáveis mesmo com esses valores ausentes.

# Questão 3

In [ ]:
df = pd.read_csv('dados/publicacoes_analise.csv')
df["taxa_utilidade_pct"] = (
    df["compartilhamentos"] + df["salvamentos"]
) / df["alcance"] * 100

mediana_tema = df.groupby("tema")["taxa_utilidade_pct"].mean().reset_index()  # média da taxa de engajamento, por dia
mediana_tema = mediana_tema.sort_values("taxa_utilidade_pct")

print(mediana_tema)


fig, ax = plt.subplots(figsize=(9, 5))  # cria a figura e os eixos, com um tamanho legível

ax.bar(mediana_tema["tema"], mediana_tema["taxa_utilidade_pct"], color="#3b6ea5")  # barra por hashtag; multiplicamos por 100 pra virar porcentagem

ax.set_title("Utilidade média por tema")  # título informativo: diz o que o gráfico mostra
ax.set_xlabel("Tema")  # rótulo do eixo horizontal
ax.set_ylabel("Utilidade média")  # rótulo do eixo vertical, com a unidade
ax.tick_params(axis="x", rotation=45)  # gira os nomes das hashtags pra não sobrepor
fig.text(0.01, -0.02, "Fonte: dados sintéticos do Festival ViraBairro (2026)", fontsize=8, color="gray")  # fonte dos dados, sempre

fig.tight_layout()  # ajusta os espaçamentos pra nada cortar
plt.show()


O tema com a maior mediana de utilidade é mobilidade, a taxa de utilidade é calculada pela soma de compartilhamentos e salvamentos e dividida pelo alcance, sendo um tipo de medida de engajamento, que indica a performance do vídeo (quanto maior, melhor),  isso indica que o tema mobilidade possui em seu centro de dispersão o maior valor dentre todos os outros temas, entretanto vale notar que a diferença da média entre os temas é pequena, mobilidade tem uma média levemente maior. Sendo assim, é válido que o time de destaque principal para este tema para testar sua performance, mas vale ressaltar que essa métrica foi retirada de uma pequena amostra de apenas 8 vídeos, e não significa necessariamente que todos os vídeos com esse tema viralizem mais, apenas dentro da amostra.

# Questão 4

In [ ]:
df2 = pd.read_csv('dados/publicacoes_analise.csv')
momento = pd.to_datetime(df["data_publicacao"])
df2["hora_publicacao"] = momento.dt.hour
df2["dia_publicacao"] = momento.dt.day

media_engajamento = df2.groupby("dia_publicacao")["taxa_engajamento_pct"].mean().reset_index()  # média da taxa de engajamento, por dia
media_engajamento = mediana_tema.sort_values("taxa_engajamento_pct")

media_engajamento = df2.groupby("dia_publicacao").agg(
    media_engajamento=("taxa_engajamento_pct", "median"),
    n_postagens=("taxa_engajamento_pct", "count"),
    alcance_total=("alcance", "sum")
).reset_index()

media_engajamento = media_engajamento.sort_values("dia_publicacao")

taxa_media_engajamento = media_engajamento.median()

print(media_engajamento)

fig, ax = plt.subplots(figsize=(9, 5))  # nova figura, um gráfico por vez

ax.plot(
    media_engajamento["dia_publicacao"],
    media_engajamento["media_engajamento"],
    marker="o",  # marca cada ponto de dado, importante quando são poucos dias
    color="#c0392b",
)

ax.set_title("Engajamento médio por dia de coleta")  # o que o gráfico mostra
ax.set_xlabel("Dia")  # rótulo do eixo horizontal
ax.set_ylabel("Engajamento médio (%)")  # rótulo do eixo vertical, com unidade
ax.tick_params(axis="x", rotation=30)  # datas legíveis
fig.text(0.01, -0.02, "Fonte: dados sintéticos do Festival ViraBairro (2026)", fontsize=8, color="gray")

fig.tight_layout()
plt.show()

reels_noite = df2[(df2["formato"] == "reel") & (df2["hora_publicacao"] > 18)]
top5_reels = reels_noite.nlargest(5, "taxa_engajamento_pct")
colunas = ["id_publicacao", "dia_publicacao", "hora", "tema", "taxa_engajamento_pct"]

print(top5_reels[colunas])

O engajamento varia muito ao passar dos dias, com muitos picos e vales, ele costuma subir, ficar uns 2-3 dias alto e cair novamente, essa tendência porém não é necessariamente a longo prazo e sim dentro da coleta. O corte de reels noturnos não prova que horário e formato causam engajamento pois várias outras variáveis deveriam ser levadas em consideração para provar isso, como público ou qualidade do post.